# ROI-aware OpenCV Axis Detection for IBCS Charts

Deze notebook is gemaakt voor jouw situatie:

- sommige afbeeldingen zijn IBCS compliant;
- sommige afbeeldingen zijn non-compliant;
- één afbeelding kan meerdere charts bevatten;
- niet elke lijn is een x-as;
- dashboards bevatten ook dividers, labels en kleine variance charts.

## Belangrijkste verbetering

De vorige aanpak zocht assen in de **hele afbeelding**.  
Deze notebook doet het slimmer:

```text
1. Detecteer eerst losse chart-regio's / plot areas
2. Run axis detection per regio
3. Zet de gevonden assen terug op de originele afbeelding
```

Waarom?

Omdat een dashboard vaak meerdere grafieken bevat. Als je alles in één keer detecteert, ziet OpenCV ook dividers, borders en andere grafieken als mogelijke as.

## Output

- Blauw = linker as bij horizontal bar charts
- Groen = onderste x-as bij vertical bar charts
- Geel = gevonden chart-regio / ROI


In [ ]:
# Install packages if needed
%pip install opencv-python matplotlib numpy pandas

In [ ]:
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("OpenCV version:", cv2.__version__)

## 1. Settings

Pas `IMAGE_PATH` aan naar jouw afbeelding.


In [ ]:
# Change this to your own image
IMAGE_PATH = Path("../Dataset/Compliant/example.png")

OUTPUT_DIR = Path("../output/roi_axis_detection")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# General mask settings
DARK_THRESHOLD = 235
SATURATION_MAX = 200

# ROI detection settings
MIN_ROI_WIDTH = 70
MIN_ROI_HEIGHT = 45
MAX_ROI_AREA_RATIO = 0.85
ROI_PADDING = 12

# Axis detection settings
BORDER_MARGIN_RATIO = 0.02

MIN_VERTICAL_LINE_HEIGHT = 18
MAX_VERTICAL_LINE_WIDTH = 12
MIN_HORIZONTAL_LINE_WIDTH = 22
MAX_HORIZONTAL_LINE_HEIGHT = 12
MAX_LINE_SPAN_RATIO = 0.90

RIGHT_SCAN_GAP = 5
RIGHT_SCAN_WIDTH = 160
MIN_RIGHT_RUN_PIXELS = 15
MIN_BAR_BAND_HEIGHT = 3

UP_SCAN_GAP = 5
UP_SCAN_HEIGHT = 140
MIN_UP_RUN_PIXELS = 15
MIN_BAR_BAND_WIDTH = 3

MIN_SUPPORTING_BARS = 2

# Filtering
MIN_AXIS_SCORE = 180
MAX_AXES_PER_ROI = 5
MAX_TOTAL_AXES = 40

## 2. Helper functions

In [ ]:

def show_image(title, image, figsize=(13, 8), cmap=None):
    plt.figure(figsize=figsize)
    if len(image.shape) == 2:
        plt.imshow(image, cmap=cmap or "gray")
    else:
        plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis("off")
    plt.show()


def load_image(path):
    image = cv2.imread(str(path))
    if image is None:
        raise FileNotFoundError(f"Image not found: {path}")
    return image


def create_dark_mask(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    saturation = hsv[:, :, 1]

    mask = np.zeros_like(gray, dtype=np.uint8)
    mask[(gray < DARK_THRESHOLD) & (saturation < SATURATION_MAX)] = 255
    return mask


def remove_outer_border(mask):
    h, w = mask.shape[:2]
    mx = int(w * BORDER_MARGIN_RATIO)
    my = int(h * BORDER_MARGIN_RATIO)

    cleaned = mask.copy()
    cleaned[:my, :] = 0
    cleaned[h-my:, :] = 0
    cleaned[:, :mx] = 0
    cleaned[:, w-mx:] = 0
    return cleaned


def merge_overlapping_boxes(boxes, overlap_threshold=0.15):
    """
    Merge boxes that overlap or are very close.
    """
    if not boxes:
        return []

    boxes = [list(b) for b in boxes]
    merged = True

    while merged:
        merged = False
        new_boxes = []
        used = [False] * len(boxes)

        for i in range(len(boxes)):
            if used[i]:
                continue

            x1, y1, x2, y2 = boxes[i]
            used[i] = True

            for j in range(i + 1, len(boxes)):
                if used[j]:
                    continue

                ax1, ay1, ax2, ay2 = x1, y1, x2, y2
                bx1, by1, bx2, by2 = boxes[j]

                inter_x1 = max(ax1, bx1)
                inter_y1 = max(ay1, by1)
                inter_x2 = min(ax2, bx2)
                inter_y2 = min(ay2, by2)

                inter_area = max(0, inter_x2 - inter_x1) * max(0, inter_y2 - inter_y1)
                area_a = max(1, (ax2 - ax1) * (ay2 - ay1))
                area_b = max(1, (bx2 - bx1) * (by2 - by1))

                # also merge if close
                close_x = abs(ax2 - bx1) < 18 or abs(bx2 - ax1) < 18
                close_y = abs(ay2 - by1) < 18 or abs(by2 - ay1) < 18

                overlap = inter_area / min(area_a, area_b)

                if overlap > overlap_threshold or (close_x and close_y):
                    x1 = min(x1, bx1)
                    y1 = min(y1, by1)
                    x2 = max(x2, bx2)
                    y2 = max(y2, by2)
                    used[j] = True
                    merged = True

            new_boxes.append([x1, y1, x2, y2])

        boxes = new_boxes

    return boxes


def detect_chart_rois(image):
    """
    Detects approximate chart regions by grouping dark chart elements.
    This is not perfect, but it helps reduce false detections by processing smaller regions.
    """
    h, w = image.shape[:2]
    mask = create_dark_mask(image)

    # Dilate to connect bars, labels and axes inside the same chart area.
    kernel_w = max(15, int(w * 0.025))
    kernel_h = max(10, int(h * 0.020))
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (kernel_w, kernel_h))
    connected = cv2.dilate(mask, kernel, iterations=2)

    contours, _ = cv2.findContours(connected, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    boxes = []

    for cnt in contours:
        x, y, bw, bh = cv2.boundingRect(cnt)

        if bw < MIN_ROI_WIDTH or bh < MIN_ROI_HEIGHT:
            continue

        if (bw * bh) > (w * h * MAX_ROI_AREA_RATIO):
            # likely whole image border/background group
            continue

        x1 = max(0, x - ROI_PADDING)
        y1 = max(0, y - ROI_PADDING)
        x2 = min(w - 1, x + bw + ROI_PADDING)
        y2 = min(h - 1, y + bh + ROI_PADDING)

        boxes.append([x1, y1, x2, y2])

    boxes = merge_overlapping_boxes(boxes)

    # Sort top-to-bottom, left-to-right
    boxes = sorted(boxes, key=lambda b: (b[1], b[0]))

    return boxes, mask, connected


def make_horizontal_structure_mask(mask):
    h, w = mask.shape[:2]
    kernel_width = max(10, int(w * 0.04))
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (kernel_width, 3))
    return cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=1)


def make_vertical_structure_mask(mask):
    h, w = mask.shape[:2]
    kernel_height = max(10, int(h * 0.04))
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3, kernel_height))
    return cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=1)


def detect_vertical_line_candidates(mask):
    h, w = mask.shape[:2]
    kernel_height = max(15, int(h * 0.18))
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1, kernel_height))
    vertical_lines_mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)

    contours, _ = cv2.findContours(vertical_lines_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    lines = []

    for cnt in contours:
        x, y, bw, bh = cv2.boundingRect(cnt)

        if bh < MIN_VERTICAL_LINE_HEIGHT:
            continue
        if bw > MAX_VERTICAL_LINE_WIDTH:
            continue
        if bh > h * MAX_LINE_SPAN_RATIO:
            continue

        lines.append({
            "axis_type": "left_axis",
            "x1": int(x + bw / 2),
            "y1": int(y),
            "x2": int(x + bw / 2),
            "y2": int(y + bh),
            "width": int(bw),
            "height": int(bh)
        })

    return lines, vertical_lines_mask


def detect_horizontal_line_candidates(mask):
    h, w = mask.shape[:2]
    kernel_width = max(15, int(w * 0.18))
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (kernel_width, 1))
    horizontal_lines_mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)

    contours, _ = cv2.findContours(horizontal_lines_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    lines = []

    for cnt in contours:
        x, y, bw, bh = cv2.boundingRect(cnt)

        if bw < MIN_HORIZONTAL_LINE_WIDTH:
            continue
        if bh > MAX_HORIZONTAL_LINE_HEIGHT:
            continue
        if bw > w * MAX_LINE_SPAN_RATIO:
            continue

        lines.append({
            "axis_type": "bottom_x_axis",
            "x1": int(x),
            "y1": int(y + bh / 2),
            "x2": int(x + bw),
            "y2": int(y + bh / 2),
            "width": int(bw),
            "height": int(bh)
        })

    return lines, horizontal_lines_mask


def count_horizontal_bar_support(line, horizontal_mask):
    h, w = horizontal_mask.shape[:2]
    x = line["x1"]
    y1 = max(0, line["y1"])
    y2 = min(h - 1, line["y2"])

    scan_x1 = min(w - 1, x + 1)
    scan_x2 = min(w, x + RIGHT_SCAN_GAP + RIGHT_SCAN_WIDTH)

    if scan_x2 <= scan_x1 or y2 <= y1:
        return 0, []

    region = horizontal_mask[y1:y2 + 1, scan_x1:scan_x2]
    row_counts = np.count_nonzero(region, axis=1)
    supported_rows = row_counts >= MIN_RIGHT_RUN_PIXELS

    bands = []
    start = None

    for idx, value in enumerate(supported_rows):
        if value and start is None:
            start = idx
        elif not value and start is not None:
            end = idx - 1
            if end - start + 1 >= MIN_BAR_BAND_HEIGHT:
                bands.append((y1 + start, y1 + end))
            start = None

    if start is not None:
        end = len(supported_rows) - 1
        if end - start + 1 >= MIN_BAR_BAND_HEIGHT:
            bands.append((y1 + start, y1 + end))

    merged = []
    for band in bands:
        if not merged:
            merged.append(list(band))
        else:
            if band[0] - merged[-1][1] <= 4:
                merged[-1][1] = band[1]
            else:
                merged.append(list(band))

    return len(merged), [(int(a), int(b)) for a, b in merged]


def count_vertical_bar_support(line, vertical_mask):
    h, w = vertical_mask.shape[:2]
    y = line["y1"]
    x1 = max(0, line["x1"])
    x2 = min(w - 1, line["x2"])

    scan_y1 = max(0, y - UP_SCAN_GAP - UP_SCAN_HEIGHT)
    scan_y2 = max(0, y - 1)

    if scan_y2 <= scan_y1 or x2 <= x1:
        return 0, []

    region = vertical_mask[scan_y1:scan_y2 + 1, x1:x2 + 1]
    col_counts = np.count_nonzero(region, axis=0)
    supported_cols = col_counts >= MIN_UP_RUN_PIXELS

    bands = []
    start = None

    for idx, value in enumerate(supported_cols):
        if value and start is None:
            start = idx
        elif not value and start is not None:
            end = idx - 1
            if end - start + 1 >= MIN_BAR_BAND_WIDTH:
                bands.append((x1 + start, x1 + end))
            start = None

    if start is not None:
        end = len(supported_cols) - 1
        if end - start + 1 >= MIN_BAR_BAND_WIDTH:
            bands.append((x1 + start, x1 + end))

    merged = []
    for band in bands:
        if not merged:
            merged.append(list(band))
        else:
            if band[0] - merged[-1][1] <= 4:
                merged[-1][1] = band[1]
            else:
                merged.append(list(band))

    return len(merged), [(int(a), int(b)) for a, b in merged]


def detect_axes_inside_roi(roi_image):
    """
    Detect axes inside one ROI. Coordinates are local to the ROI.
    """
    dark_mask = create_dark_mask(roi_image)
    clean_mask = remove_outer_border(dark_mask)

    hmask = make_horizontal_structure_mask(clean_mask)
    vmask = make_vertical_structure_mask(clean_mask)

    vertical_lines, _ = detect_vertical_line_candidates(clean_mask)
    horizontal_lines, _ = detect_horizontal_line_candidates(clean_mask)

    axes = []

    for line in vertical_lines:
        support_count, bands = count_horizontal_bar_support(line, hmask)
        if support_count >= MIN_SUPPORTING_BARS:
            y1 = min(b[0] for b in bands)
            y2 = max(b[1] for b in bands)
            x = line["x1"]
            pad = int((y2 - y1) * 0.12)
            y1 = max(0, y1 - pad)
            y2 = min(roi_image.shape[0] - 1, y2 + pad)

            score = support_count * 120 + (y2 - y1) + line["height"] * 0.25
            if score >= MIN_AXIS_SCORE:
                axes.append({
                    "axis_type": "left_axis",
                    "x1": int(x),
                    "y1": int(y1),
                    "x2": int(x),
                    "y2": int(y2),
                    "supporting_bars": int(support_count),
                    "score": float(score)
                })

    for line in horizontal_lines:
        support_count, bands = count_vertical_bar_support(line, vmask)
        if support_count >= MIN_SUPPORTING_BARS:
            x1 = min(b[0] for b in bands)
            x2 = max(b[1] for b in bands)
            y = line["y1"]
            pad = int((x2 - x1) * 0.12)
            x1 = max(0, x1 - pad)
            x2 = min(roi_image.shape[1] - 1, x2 + pad)

            score = support_count * 120 + (x2 - x1) + line["width"] * 0.25
            if score >= MIN_AXIS_SCORE:
                axes.append({
                    "axis_type": "bottom_x_axis",
                    "x1": int(x1),
                    "y1": int(y),
                    "x2": int(x2),
                    "y2": int(y),
                    "supporting_bars": int(support_count),
                    "score": float(score)
                })

    axes = sorted(axes, key=lambda a: a["score"], reverse=True)
    return axes[:MAX_AXES_PER_ROI]


def remove_duplicate_global_axes(axes, tolerance=10):
    if not axes:
        return []

    axes = sorted(axes, key=lambda a: a["score"], reverse=True)
    kept = []

    for axis in axes:
        duplicate = False

        for other in kept:
            if axis["axis_type"] != other["axis_type"]:
                continue

            if axis["axis_type"] == "left_axis":
                close_x = abs(axis["x1"] - other["x1"]) <= tolerance
                overlap_y = min(axis["y2"], other["y2"]) - max(axis["y1"], other["y1"])
                if close_x and overlap_y > 0:
                    duplicate = True
                    break

            if axis["axis_type"] == "bottom_x_axis":
                close_y = abs(axis["y1"] - other["y1"]) <= tolerance
                overlap_x = min(axis["x2"], other["x2"]) - max(axis["x1"], other["x1"])
                if close_y and overlap_x > 0:
                    duplicate = True
                    break

        if not duplicate:
            kept.append(axis)

    return kept


def run_roi_axis_detection(image):
    rois, mask, connected = detect_chart_rois(image)
    all_axes = []

    for roi_id, (x1, y1, x2, y2) in enumerate(rois, start=1):
        roi = image[y1:y2, x1:x2]
        local_axes = detect_axes_inside_roi(roi)

        for axis in local_axes:
            global_axis = axis.copy()
            global_axis["roi_id"] = roi_id
            global_axis["roi_box"] = [x1, y1, x2, y2]
            global_axis["x1"] += x1
            global_axis["x2"] += x1
            global_axis["y1"] += y1
            global_axis["y2"] += y1
            all_axes.append(global_axis)

    all_axes = remove_duplicate_global_axes(all_axes)
    all_axes = sorted(all_axes, key=lambda a: a["score"], reverse=True)
    return {
        "rois": rois,
        "mask": mask,
        "connected": connected,
        "axes": all_axes[:MAX_TOTAL_AXES]
    }


def draw_roi_axis_result(image, result):
    output = image.copy()

    # draw ROIs in yellow
    for i, (x1, y1, x2, y2) in enumerate(result["rois"], start=1):
        cv2.rectangle(output, (x1, y1), (x2, y2), (0, 220, 220), 2)
        cv2.putText(output, f"ROI {i}", (x1, max(15, y1 - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 220, 220), 1)

    # draw axes
    for i, axis in enumerate(result["axes"], start=1):
        x1, y1, x2, y2 = axis["x1"], axis["y1"], axis["x2"], axis["y2"]

        if axis["axis_type"] == "left_axis":
            color = (255, 0, 0)
            label = f"L{i} r{axis['roi_id']} b={axis['supporting_bars']}"
            text_pos = (x1 + 5, max(15, y1 + 15))
        else:
            color = (0, 180, 0)
            label = f"B{i} r{axis['roi_id']} b={axis['supporting_bars']}"
            text_pos = (x1, max(15, y1 - 8))

        cv2.line(output, (x1, y1), (x2, y2), color, 3)
        cv2.putText(output, label, text_pos, cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

    return output


def axes_to_dataframe(axes):
    rows = []
    for i, a in enumerate(axes, start=1):
        rows.append({
            "rank": i,
            "roi_id": a.get("roi_id"),
            "axis_type": a.get("axis_type"),
            "x1": a.get("x1"),
            "y1": a.get("y1"),
            "x2": a.get("x2"),
            "y2": a.get("y2"),
            "supporting_bars": a.get("supporting_bars"),
            "score": round(a.get("score", 0), 1),
            "roi_box": a.get("roi_box")
        })
    return pd.DataFrame(rows)


## 3. Run ROI-aware detection on one image

In [ ]:
image = load_image(IMAGE_PATH)
show_image("Original image", image)

result = run_roi_axis_detection(image)

show_image("Dark mask", result["mask"])
show_image("Connected chart areas for ROI detection", result["connected"])

print("ROIs found:", len(result["rois"]))
print("Axes found:", len(result["axes"]))

output = draw_roi_axis_result(image, result)
show_image("ROI-aware axis detection result", output)

output_path = OUTPUT_DIR / f"{IMAGE_PATH.stem}_roi_axis_detection.png"
cv2.imwrite(str(output_path), output)

print("Saved result to:", output_path)

display(axes_to_dataframe(result["axes"]))

## 4. Batch test on dataset

Deze cel runt de ROI-aware detectie op alle afbeeldingen in `Dataset/Compliant` en `Dataset/Non-Compliant`.


In [ ]:
DATASET_DIR = Path("../Dataset")
BATCH_OUTPUT_DIR = OUTPUT_DIR / "batch"
BATCH_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

image_extensions = [".png", ".jpg", ".jpeg", ".webp"]
image_paths = []

for ext in image_extensions:
    image_paths.extend(DATASET_DIR.rglob(f"*{ext}"))

print("Found images:", len(image_paths))

summary = []

for path in image_paths:
    try:
        img = load_image(path)
        res = run_roi_axis_detection(img)
        out = draw_roi_axis_result(img, res)

        relative = path.relative_to(DATASET_DIR)
        save_path = BATCH_OUTPUT_DIR / relative
        save_path.parent.mkdir(parents=True, exist_ok=True)

        cv2.imwrite(str(save_path), out)

        summary.append({
            "image": str(relative),
            "rois_found": len(res["rois"]),
            "axes_found": len(res["axes"]),
            "left_axes": sum(1 for a in res["axes"] if a["axis_type"] == "left_axis"),
            "bottom_x_axes": sum(1 for a in res["axes"] if a["axis_type"] == "bottom_x_axis"),
            "output": str(save_path)
        })

        print("Saved:", save_path)

    except Exception as e:
        print("Error:", path, e)
        summary.append({
            "image": str(path),
            "error": str(e)
        })

summary_df = pd.DataFrame(summary)
summary_path = OUTPUT_DIR / "roi_axis_detection_summary.csv"
summary_df.to_csv(summary_path, index=False)

display(summary_df)
print("Summary saved to:", summary_path)

## 5. What to write in your report

You can use this explanation:

> The dataset contains complex IBCS chart images where one image may include multiple chart areas, horizontal bar charts, vertical bar charts, variance charts, dividers, borders and text labels. A global axis detection approach produced many false positives, because OpenCV detected lines from the whole image at once. Therefore, an ROI-aware approach was used. First, dark chart elements are grouped to detect approximate chart regions. Then axis detection is applied separately inside each region. For horizontal bar charts, a vertical line is accepted as a left axis when multiple horizontal bar segments start to its right. For vertical bar charts, a horizontal line is accepted as a bottom x-axis when multiple vertical bar segments end above it. The detected local axes are then converted back to the coordinates of the original image.

Limitatie:

> This is still a rule-based OpenCV method. It works better than global line detection because it processes smaller chart regions, but it can still fail when chart regions are merged, when bars are very small, or when text and dividers look similar to bars and axes. A YOLO model could be added later to detect chart regions or axes more robustly.
